In [2]:
# from flask import Flask, render_template, request, jsonify
# import pytesseract
# import cv2
# import re
# import os
# import pandas as pd
# import numpy as np
# from PIL import Image
# from fuzzywuzzy import fuzz
# import warnings
# import base64
# import uuid
# warnings.filterwarnings('ignore')

# app = Flask(__name__)
# app.config['UPLOAD_FOLDER'] = 'uploads'
# app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB max

# os.makedirs('uploads', exist_ok=True)

# # -----------------------------------------------
# # Tesseract path — update if needed
# # -----------------------------------------------
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# # -----------------------------------------------
# # MODULE 1: Preprocess Image
# # -----------------------------------------------
# def preprocess_image(image_path):
#     img = cv2.imread(image_path)
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
#     avg_brightness = np.mean(gray)
#     if avg_brightness < 127:
#         gray = cv2.bitwise_not(gray)
#     _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
#     kernel = np.ones((1, 1), np.uint8)
#     thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
#     return thresh

# # -----------------------------------------------
# # MODULE 2: Extract and Clean Text
# # -----------------------------------------------
# def extract_text(image_path):
#     preprocessed = preprocess_image(image_path)
#     pil_img = Image.fromarray(preprocessed)
#     custom_config = r'--oem 3 --psm 6'
#     raw_text = pytesseract.image_to_string(pil_img, config=custom_config)
#     return raw_text

# def clean_text(raw_text):
#     text = raw_text.lower()
#     text = re.sub(r'[^a-z0-9\s]', ' ', text)
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

# # -----------------------------------------------
# # MODULE 3: Fuzzy Match
# # -----------------------------------------------
# def fuzzy_match(cleaned_text, df, threshold=50):
#     results = []
#     unique_products = df.drop_duplicates(subset='product_name')
#     for _, row in unique_products.iterrows():
#         product_lower = row['product_name'].lower()
#         brand_lower   = row['brand'].lower()
#         name_score    = fuzz.partial_ratio(cleaned_text, product_lower)
#         brand_score   = fuzz.partial_ratio(cleaned_text, brand_lower)
#         best_score    = max(name_score, brand_score)
#         if best_score >= threshold:
#             results.append({
#                 'product_name': row['product_name'],
#                 'brand':        row['brand'],
#                 'category':     row['category'],
#                 'match_score':  best_score
#             })
#     results = sorted(results, key=lambda x: x['match_score'], reverse=True)
#     return results[:5]

# # -----------------------------------------------
# # MODULE 4: Get Prices
# # -----------------------------------------------
# def get_prices(product_name, df):
#     matched = df[df['product_name'].str.lower() == product_name.lower()]
#     prices  = matched[['seller', 'price_NPR', 'in_stock']].sort_values('price_NPR')
#     return prices.to_dict('records')

# # -----------------------------------------------
# # ROUTES
# # -----------------------------------------------
# @app.route('/')
# def index():
#     return render_template('index.html')

# @app.route('/analyze', methods=['POST'])
# def analyze():
#     if 'image' not in request.files:
#         return jsonify({'error': 'No image uploaded'}), 400

#     file = request.files['image']
#     if file.filename == '':
#         return jsonify({'error': 'No file selected'}), 400

#     # Save uploaded image
#     ext      = os.path.splitext(file.filename)[1]
#     filename = f"{uuid.uuid4()}{ext}"
#     filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
#     file.save(filepath)

#     try:
#         # Step 1: Extract text
#         raw_text   = extract_text(filepath)
#         clean      = clean_text(raw_text)

#         if not clean.strip():
#             return jsonify({'error': 'Could not extract text from image. Please use a clearer product image.'}), 400

#         # Step 2: Load dataset
#         df = pd.read_csv('products_all.csv')

#         # Step 3: Match product
#         matches = fuzzy_match(clean, df, threshold=50)

#         if not matches:
#             return jsonify({'error': 'No matching product found. Try a clearer image.'}), 400

#         top_match = matches[0]

#         # Step 4: Get prices
#         prices = get_prices(top_match['product_name'], df)

#         # Step 5: Build response
#         result = {
#             'ocr_text':     clean[:100],
#             'product_name': top_match['product_name'],
#             'brand':        top_match['brand'],
#             'category':     top_match['category'],
#             'match_score':  top_match['match_score'],
#             'prices':       prices,
#             'best_price':   prices[0]['price_NPR'] if prices else 0,
#             'best_seller':  prices[0]['seller'] if prices else '',
#             'worst_price':  prices[-1]['price_NPR'] if prices else 0,
#             'you_save':     round(prices[-1]['price_NPR'] - prices[0]['price_NPR'], 2) if prices else 0,
#             'other_matches': matches[1:4]
#         }

#         return jsonify(result)

#     except Exception as e:
#         return jsonify({'error': str(e)}), 500

#     finally:
#         # Clean up uploaded file
#         if os.path.exists(filepath):
#             os.remove(filepath)

# if __name__ == '__main__':
#     app.run(debug=True)


In [3]:
from flask import Flask, request, jsonify, render_template
import pytesseract
from PIL import Image
import cv2
import numpy as np
import pandas as pd
import pickle
from fuzzywuzzy import fuzz
import os
import warnings
warnings.filterwarnings('ignore')

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("Libraries loaded!")

Libraries loaded!


In [4]:
# Load ML model and vectorizer
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

# Load dataset
df = pd.read_csv('products_all_fixed.csv')

print("Model loaded!")
print("Vectorizer loaded!")
print(f"Dataset loaded — {len(df)} rows")

Model loaded!
Vectorizer loaded!
Dataset loaded — 8000 rows


In [5]:
def preprocess_image(img):
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Resize 2x for better OCR
    gray = cv2.resize(gray, None, fx=2, fy=2,
                      interpolation=cv2.INTER_CUBIC)

    # Apply OTSU thresholding
    _, thresh = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Invert if image is dark
    if np.mean(thresh) < 127:
        thresh = cv2.bitwise_not(thresh)

    return thresh

print("Preprocess function ready!")

Preprocess function ready!


##  OCR Text Extraction Function

In [6]:
def extract_text(processed_img):
    pil_img = Image.fromarray(processed_img)
    config = '--oem 3 --psm 6'
    text = pytesseract.image_to_string(pil_img, config=config)

    # Clean text
    text = text.lower().strip()
    text = ''.join(c if c.isalnum() or c == ' ' else ' ' for c in text)
    text = ' '.join(text.split())

    return text

print("OCR function ready!")

OCR function ready!


In [7]:
# ML Category Prediction Function
def predict_category(text):
    text_vec = vectorizer.transform([text])
    category = model.predict(text_vec)[0]
    confidence = round(model.predict_proba(text_vec).max() * 100, 1)
    return category, confidence

print("ML prediction function ready!")

ML prediction function ready!


In [8]:
# Fuzzy Product Matching Function
def match_product(text, category):
    # Filter dataset by predicted category
    df_filtered = df[df['category'] == category]

    scores = []
    for product in df_filtered['product_name'].unique():
        score = fuzz.partial_ratio(text.lower(), product.lower())
        scores.append((product, score))

    # Sort by score descending
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Return top match
    best_match, best_score = scores[0]
    return best_match, best_score

print("Fuzzy matching function ready!")

Fuzzy matching function ready!


In [9]:
# Price Comparison Function
def compare_prices(product_name):
    # Get all rows for matched product
    product_df = df[df['product_name'] == product_name].copy()

    # Sort by price low to high
    product_df = product_df.sort_values('price_NPR')

    results = []
    for _, row in product_df.iterrows():
        results.append({
            'seller':    row['seller'],
            'price_NPR': round(row['price_NPR'], 2),
        })

    best_price  = product_df['price_NPR'].min()
    worst_price = product_df['price_NPR'].max()
    you_save    = round(worst_price - best_price, 2)

    return results, you_save

print("Price comparison function ready!")

Price comparison function ready!


Flask App & Routes

In [10]:
app = Flask(__name__)
UPLOAD_FOLDER = 'uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/analyze', methods=['POST'])
def analyze():
    try:
        # Step 1 — Get uploaded image
        file = request.files['image']
        filepath = os.path.join(UPLOAD_FOLDER, file.filename)
        file.save(filepath)

        # Step 2 — Preprocess image
        img = cv2.imread(filepath)
        processed = preprocess_image(img)

        # Step 3 — Extract text via OCR
        ocr_text = extract_text(processed)

        if not ocr_text:
            return jsonify({'error': 'Could not extract text from image.'}), 400

        # Step 4 — ML predicts category
        category, confidence = predict_category(ocr_text)

        # Step 5 — Fuzzy match product
        matched_product, match_score = match_product(ocr_text, category)

        # Step 6 — Compare prices
        prices, you_save = compare_prices(matched_product)

        # Step 7 — Return results
        return jsonify({
            'ocr_text':        ocr_text,
            'category':        category,
            'confidence':      confidence,
            'matched_product': matched_product,
            'match_score':     match_score,
            'prices':          prices,
            'you_save':        you_save,
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

print("Flask routes ready!")

Flask routes ready!


In [12]:
# This library fixes the conflict between Jupyter and Flask.
import nest_asyncio
nest_asyncio.apply()
# Run the App

app.run(debug=False, use_reloader=False, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [09/Mar/2026 13:56:00] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [09/Mar/2026 13:56:36] "POST /analyze HTTP/1.1" 200 -
127.0.0.1 - - [09/Mar/2026 13:56:44] "POST /analyze HTTP/1.1" 200 -
